# Smart Patio Shield - ConvLSTM and the diurnal control (Notebook 08)

Two questions raised in supervision, tested in one session.

**Q1 - Does the architecture matter, or was the sampling interval the problem?**
Notebook 07 stacked consecutive hourly frames as extra channels and performance
declined (test PR-AUC 0.4034 -> 0.3809 -> 0.3638 for n = 1, 2, 3). Channel stacking
throws away temporal order: a convolution over stacked channels cannot tell that one
frame precedes another. A ConvLSTM keeps the spatial map and carries a recurrent state
across frames, so it is a genuinely different test rather than a rerun.

**Q2 - Is any apparent motion signal really a time-of-day signal?**
Rainfall here has a strong diurnal cycle (roughly 1.8% positive at 06:00 against 31.1%
at 14:00) and the pressure record shows the semidiurnal atmospheric tide. A model can
score well by learning "it is mid-afternoon" without learning anything about motion.
The control: give each frame its own diurnal (24 h) and semidiurnal (12 h) harmonics as
explicit input planes. Any remaining benefit from recurrence must then be information
beyond timing.

**Outcome**. Recurrence recovered most of what channel stacking lost (0.3638 → 0.4013 at three frames) without exceeding the single-frame reference of 0.4034, so the sampling interval remains the binding constraint. Supplying explicit timing raised the model further to 0.4268, which was not the anticipated direction for a control and indicates that time of day carries information infrared imagery does not. Fusion with the stronger branch reached 0.7303, still short of the tabular model.

Sections
- **0** Session setup
- **1** Datasets, channel-order verification, time-harmonic planes
- **2** Train ConvLSTM, with and without the diurnal control
- **3** Stratified evaluation by time of day
- **4** Fusion with the ConvLSTM vision branch
- **5** Save results

## Section 0 - Session setup
Same idempotent setup as notebooks 05 and 07. Requires `src/models/goes_temporal.py`
and `src/models/convlstm.py` to be committed and pushed.

In [1]:
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - Runtime > Change runtime type > T4 GPU")
if not os.path.exists("/content/smart-patio-shield"):
    from getpass import getpass
    tok = getpass("GitHub token: ")
    !git clone https://{tok}@github.com/romayneg/smart-patio-shield.git /content/smart-patio-shield
%cd /content/smart-patio-shield
!git pull -q
print("code up to date")

GPU: NVIDIA A100-SXM4-40GB
GitHub token: ··········
Cloning into '/content/smart-patio-shield'...
remote: Enumerating objects: 170, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 170 (delta 14), reused 23 (delta 10), pack-reused 122 (from 1)
Receiving objects: 100% (170/170), 88.21 MiB | 17.50 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/smart-patio-shield
code up to date


In [2]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/smart-patio-shield"

import shutil
def relink(target, linkname):
    if os.path.islink(linkname):   os.unlink(linkname)
    elif os.path.isdir(linkname):  shutil.rmtree(linkname)
    elif os.path.exists(linkname): os.remove(linkname)
    parent = os.path.dirname(linkname)
    if parent: os.makedirs(parent, exist_ok=True)
    os.symlink(target, linkname)

relink(f"{DRIVE}/data/goes",                   "data/raw/goes")
relink(f"{DRIVE}/data/image_labels.parquet",   "data/processed/image_labels.parquet")
relink(f"{DRIVE}/data/patio_features.parquet", "data/processed/patio_features.parquet")
relink(f"{DRIVE}/models",                      "models")
print("day-files:", len(os.listdir("data/raw/goes")))
for f in ["src/models/goes_temporal.py", "src/models/convlstm.py"]:
    assert os.path.exists(f), f"{f} not found - commit & push it, then re-run Section 0."
print("setup complete")

Mounted at /content/drive
day-files: 1827
setup complete


## Section 1 - Datasets, channel order, and time-harmonic planes

Same fairness control as notebook 07: the n = 3 keys define the common sample set and
every configuration is restricted to those identical hours, so nothing is compared
against an easier sample.

The channel-order check below matters. `ConvLSTMClassifier` reshapes `(C*T, H, W)` into
`(T, C, H, W)`, which is only correct if frames sit in contiguous blocks, oldest first.
Rather than trust that, the cell verifies it: with shared normalisation statistics, the
final frame-block of an n = 3 tensor must equal the n = 1 tensor for the same hour. It
fails loudly if the layout differs.

In [3]:
import importlib, numpy as np, torch
import src.models.goes_temporal as gt
import src.models.goes_dataset as gd
import src.models.convlstm as cl
for m in (gt, gd, cl): importlib.reload(m)

patches = gd._load_all_patches()
print(f"patches in memory: {len(patches):,}")

BANDS = gt.IR_ONLY      # isolate motion from the C02 day/night confound
CPF   = len(BANDS)      # channels per frame
NF    = 3               # frames per sample

def build(split, n_frames, norm=None, restrict_keys=None):
    ds = gt.GoesTemporalDataset(split, channels=BANDS, n_frames=n_frames,
                                norm_stats=norm, patches=patches)
    if restrict_keys is not None:
        keep = set(restrict_keys)
        ds.keys = [k for k in ds.keys if k in keep]
    return ds

common = {}
for split in ["train", "val", "test"]:
    common[split] = set(build(split, NF).keys)
    print(f"{split}: common (n={NF}-eligible) samples = {len(common[split]):,}")

patches in memory: 87,648
train: common (n=3-eligible) samples = 61,382
val: common (n=3-eligible) samples = 13,154
test: common (n=3-eligible) samples = 13,108


In [4]:
# Verify frame-major channel ordering before trusting the reshape.
probe3 = build("val", NF, restrict_keys=common["val"])
NORM_PROBE = probe3.norm_stats
probe1 = build("val", 1, norm=NORM_PROBE, restrict_keys=common["val"])

k = probe3.keys[0]
x3 = probe3[probe3.keys.index(k)][0].numpy()
x1 = probe1[probe1.keys.index(k)][0].numpy()
print("n=3 tensor:", x3.shape, " n=1 tensor:", x1.shape)

last_block = x3[-CPF:]
if np.allclose(last_block, x1, atol=1e-5):
    print("PASS - frames are contiguous blocks, oldest first; the reshape is correct.")
else:
    diffs = [float(np.abs(x3[i*CPF:(i+1)*CPF] - x1).mean()) for i in range(NF)]
    print("FAIL - last block does not match the single-frame tensor.")
    print("mean abs diff by block:", [round(d, 5) for d in diffs])
    raise AssertionError("Channel layout differs from the frame-major assumption. "
                         "Check goes_temporal.py before training.")

n=3 tensor: (6, 64, 64)  n=1 tensor: (2, 64, 64)
PASS - frames are contiguous blocks, oldest first; the reshape is correct.


In [5]:
from torch.utils.data import Dataset
from datetime import datetime

class TimeHarmonics(Dataset):
    """Append per-frame diurnal (24 h) and semidiurnal (12 h) harmonics.

    Each frame receives the harmonics for its own hour, so frame t-2 is labelled two
    hours earlier than the current frame. With timing supplied explicitly, any gain
    from recurrence has to be information the clock does not already provide.
    """

    def __init__(self, base, n_frames, cpf):
        self.base, self.n_frames, self.cpf = base, n_frames, cpf
        self.keys, self.label_of = base.keys, base.label_of
        self.extra = 4

    def __len__(self):
        return len(self.base)

    @staticmethod
    def _hour(key):
        return datetime.fromisoformat(key.split("|", 1)[1]).hour

    def __getitem__(self, i):
        x, y = self.base[i]
        h0 = self._hour(self.keys[i])
        hgt, wid = x.shape[-2:]
        blocks = []
        for j in range(self.n_frames):
            h = (h0 - (self.n_frames - 1 - j)) % 24
            vals = [np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24),
                    np.sin(4*np.pi*h/24), np.cos(4*np.pi*h/24)]
            planes = torch.stack([torch.full((hgt, wid), float(v)) for v in vals])
            blocks.append(torch.cat([x[j*self.cpf:(j+1)*self.cpf], planes], dim=0))
        return torch.cat(blocks, dim=0), y

print("TimeHarmonics ready: adds 4 planes per frame (24 h and 12 h sin/cos).")

TimeHarmonics ready: adds 4 planes per frame (24 h and 12 h sin/cos).


## Section 2 - Train the ConvLSTM, with and without the diurnal control

Two configurations, identical apart from the timing planes:

- **A - ConvLSTM (plain)**: 2 IR channels per frame, 3 frames.
- **B - ConvLSTM + time harmonics**: 6 channels per frame (2 IR + 4 timing).

Training recipe matches `cnn.train_model`: class-weighted BCE, Adam, PR-AUC early
stopping. Learning rate is 1e-3 rather than 1e-4 because this network is trained from
scratch instead of fine-tuned.

In [6]:
from torch.utils.data import DataLoader

train_base = build("train", NF, restrict_keys=common["train"])
NORM = train_base.norm_stats
val_base  = build("val",  NF, norm=NORM, restrict_keys=common["val"])
test_base = build("test", NF, norm=NORM, restrict_keys=common["test"])
print(f"samples train/val/test: {len(train_base)}/{len(val_base)}/{len(test_base)}")

results = {}

print("\n" + "="*30 + "  A: ConvLSTM (plain)  " + "="*30)
model_a, hist_a = cl.train_convlstm(train_base, val_base, channels_per_frame=CPF,
                                    n_frames=NF, hidden=64, epochs=25,
                                    batch_size=64, lr=1e-3, patience=4)
dev = "cuda" if torch.cuda.is_available() else "cpu"
ev_a = cl.evaluate(model_a, DataLoader(test_base, batch_size=128, num_workers=2), dev)
results["convlstm_plain"] = {"val_pr_auc": round(max(h["val_pr_auc"] for h in hist_a), 4),
                             "test_pr_auc": round(ev_a["pr_auc"], 4),
                             "channels_per_frame": CPF, "n_frames": NF}
print(f"A test PR-AUC: {ev_a['pr_auc']:.4f}")

samples train/val/test: 61382/13154/13108

==============================  A: ConvLSTM (plain)  ==============================
device cuda | 2ch x 3 frames | pos_weight 10.09 | params 152,513
  epoch  0  loss 1.1677  val PR-AUC 0.2172
  epoch  1  loss 1.1195  val PR-AUC 0.2323
  epoch  2  loss 1.0576  val PR-AUC 0.2480
  epoch  3  loss 1.0225  val PR-AUC 0.2679
  epoch  4  loss 1.0093  val PR-AUC 0.2798
  epoch  5  loss 0.9966  val PR-AUC 0.2622
  epoch  6  loss 0.9919  val PR-AUC 0.2929
  epoch  7  loss 0.9802  val PR-AUC 0.2967
  epoch  8  loss 0.9716  val PR-AUC 0.2973
  epoch  9  loss 0.9651  val PR-AUC 0.3119
  epoch 10  loss 0.9497  val PR-AUC 0.2953
  epoch 11  loss 0.9408  val PR-AUC 0.3352
  epoch 12  loss 0.9270  val PR-AUC 0.3178
  epoch 13  loss 0.9201  val PR-AUC 0.3114
  epoch 14  loss 0.9101  val PR-AUC 0.3427
  epoch 15  loss 0.9028  val PR-AUC 0.3483
  epoch 16  loss 0.8893  val PR-AUC 0.3452
  epoch 17  loss 0.8868  val PR-AUC 0.3550
  epoch 18  loss 0.8829  val PR-AU

In [7]:
print("\n" + "="*26 + "  B: ConvLSTM + time harmonics  " + "="*26)
train_t = TimeHarmonics(train_base, NF, CPF)
val_t   = TimeHarmonics(val_base,   NF, CPF)
test_t  = TimeHarmonics(test_base,  NF, CPF)

model_b, hist_b = cl.train_convlstm(train_t, val_t, channels_per_frame=CPF + 4,
                                    n_frames=NF, hidden=64, epochs=25,
                                    batch_size=64, lr=1e-3, patience=4)
ev_b = cl.evaluate(model_b, DataLoader(test_t, batch_size=128, num_workers=2), dev)
results["convlstm_time_controlled"] = {
    "val_pr_auc": round(max(h["val_pr_auc"] for h in hist_b), 4),
    "test_pr_auc": round(ev_b["pr_auc"], 4),
    "channels_per_frame": CPF + 4, "n_frames": NF}
print(f"B test PR-AUC: {ev_b['pr_auc']:.4f}")

print("\n==== ConvLSTM summary (shared samples, IR-only) ====")
print(f"{'config':<28} {'val':>8} {'test':>8}")
for k, v in results.items():
    print(f"{k:<28} {v['val_pr_auc']:>8} {v['test_pr_auc']:>8}")
print("\nReference points on this same restricted sample set:")
print("  channel-stacked n=3 (notebook 07): test 0.3638")
print("  channel-stacked n=1 (notebook 07): test 0.4034")


==========================  B: ConvLSTM + time harmonics  ==========================
device cuda | 6ch x 3 frames | pos_weight 10.09 | params 161,729
  epoch  0  loss 0.9002  val PR-AUC 0.3409
  epoch  1  loss 0.8622  val PR-AUC 0.3316
  epoch  2  loss 0.8502  val PR-AUC 0.3477
  epoch  3  loss 0.8492  val PR-AUC 0.3326
  epoch  4  loss 0.8430  val PR-AUC 0.3479
  epoch  5  loss 0.8431  val PR-AUC 0.3431
  epoch  6  loss 0.8387  val PR-AUC 0.3406
  epoch  7  loss 0.8328  val PR-AUC 0.3490
  epoch  8  loss 0.8321  val PR-AUC 0.3439
  epoch  9  loss 0.8296  val PR-AUC 0.3536
  epoch 10  loss 0.8285  val PR-AUC 0.3585
  epoch 11  loss 0.8247  val PR-AUC 0.3644
  epoch 12  loss 0.8250  val PR-AUC 0.3615
  epoch 13  loss 0.8206  val PR-AUC 0.3569
  epoch 14  loss 0.8175  val PR-AUC 0.3673
  epoch 15  loss 0.8191  val PR-AUC 0.3455
  epoch 16  loss 0.8159  val PR-AUC 0.3607
  epoch 17  loss 0.8133  val PR-AUC 0.3563
  epoch 18  loss 0.8120  val PR-AUC 0.3715
  epoch 19  loss 0.8113  val PR-

Both recurrent configurations exceed the channel-stacked three-frame result of 0.3638, confirming that part of the earlier decline came from discarding temporal order rather than from the imagery itself. Neither exceeds the single-frame reference of 0.4034 by a margin worth claiming: the plain ConvLSTM lands just below it at 0.4013, and the gain in configuration B comes from the timing planes rather than from motion. Validation scores sit below test scores in both runs, which is unusual and suggests these estimates are loosely pinned; the ordering is more reliable than the individual figures.

## Section 3 - Stratified evaluation by time of day

An aggregate score can conceal an effect confined to convective hours, so PR-AUC is computed within four time-of-day windows, holding the diurnal cycle roughly constant inside each. The model exceeds the local base rate in every window, and by the widest margin overnight, where the base rate is lowest. The narrowest margin falls in the afternoon, where the base rate of 0.270 leaves least headroom. The vision signal is therefore not reducible to a clock, though the afternoon result shows how much of an aggregate score the diurnal cycle can account for.

In [8]:
from sklearn.metrics import average_precision_score
from datetime import datetime
import pandas as pd

def hour_of(key):
    return datetime.fromisoformat(key.split("|", 1)[1]).hour

BINS = [("night 00-05", range(0, 6)), ("morning 06-11", range(6, 12)),
        ("afternoon 12-17", range(12, 18)), ("evening 18-23", range(18, 24))]

hours = np.array([hour_of(k) for k in test_base.keys])
rows = []
for name, hrs in BINS:
    m = np.isin(hours, list(hrs))
    if m.sum() < 50 or ev_a["labels"][m].sum() < 5:
        rows.append({"window": name, "n": int(m.sum()), "positives": int(ev_a["labels"][m].sum()),
                     "base_rate": None, "convlstm": None, "time_controlled": None})
        continue
    rows.append({
        "window": name,
        "n": int(m.sum()),
        "positives": int(ev_a["labels"][m].sum()),
        "base_rate": round(float(ev_a["labels"][m].mean()), 4),
        "convlstm": round(float(average_precision_score(ev_a["labels"][m], ev_a["probs"][m])), 4),
        "time_controlled": round(float(average_precision_score(ev_b["labels"][m], ev_b["probs"][m])), 4),
    })

strat = pd.DataFrame(rows)
print(strat.to_string(index=False))
print("\nCompare each score against the base rate in the same row: that is the honest")
print("floor once time of day is held constant.")
results["stratified_by_time_of_day"] = rows

         window    n  positives  base_rate  convlstm  time_controlled
    night 00-05 3276        115     0.0351    0.3997           0.4088
  morning 06-11 3276        155     0.0473    0.3295           0.3214
afternoon 12-17 3276        885     0.2701    0.4178           0.4234
  evening 18-23 3280        160     0.0488    0.3553           0.3743

Compare each score against the base rate in the same row: that is the honest
floor once time of day is held constant.


## Section 4 - Fusion with the ConvLSTM vision branch

The better of A and B is carried into late fusion against the tabular branch.

One point of care: this evaluation runs on the n = 3-eligible subset, not the full test
set, so Model 1's headline 0.7557 is not the right comparator here. The tabular branch is
rescored on exactly these hours and that number is the reference.



In [9]:
import src.models.fusion as fus
importlib.reload(fus)

BEST = "convlstm_time_controlled" if (
    results["convlstm_time_controlled"]["test_pr_auc"] >
    results["convlstm_plain"]["test_pr_auc"]) else "convlstm_plain"
best_model = model_b if BEST == "convlstm_time_controlled" else model_a
best_val_ds  = val_t  if BEST == "convlstm_time_controlled" else val_base
best_test_ds = test_t if BEST == "convlstm_time_controlled" else test_base
print("carrying forward:", BEST)

def vision_probs(model, ds, keys):
    loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=2)
    ev = cl.evaluate(model, loader, dev)
    return pd.DataFrame({"key": keys, "p_img": ev["probs"], "y": ev["labels"]})

img_val  = vision_probs(best_model, best_val_ds,  val_base.keys)
img_test = vision_probs(best_model, best_test_ds, test_base.keys)

tab = fus.tabular_branch("models/xgboost_baseline.json",
                         "models/baseline_training.manifest.json")

def align(tab_frame, img_frame):
    m = tab_frame.merge(img_frame[["key", "p_img", "y"]], on="key", how="inner",
                        suffixes=("_tab", ""))
    ycol = "y" if "y" in m.columns else "y_tab"
    return {"y": m[ycol].to_numpy(), "p_tab": m["p_tab"].to_numpy(),
            "p_img": m["p_img"].to_numpy(), "n": len(m)}

val_al, test_al = align(tab["val"][0], img_val), align(tab["test"][0], img_test)
print(f"aligned val/test rows: {val_al['n']}/{test_al['n']}")

p_late, _ = fus.late_fusion(val_al, test_al)
tab_here = average_precision_score(test_al["y"], test_al["p_tab"])

print("\n==== Fusion on the n=3-eligible subset ====")
print(f"{'branch':<34} {'test PR-AUC':>12}")
print(f"{'Model 1 tabular (this subset)':<34} {tab_here:>12.4f}")
print(f"{'ConvLSTM vision':<34} {average_precision_score(test_al['y'], test_al['p_img']):>12.4f}")
print(f"{'Late fusion':<34} {average_precision_score(test_al['y'], p_late):>12.4f}")
print("\nModel 1 on the full test set is 0.7557; the subset figure above is the")
print("like-for-like comparator for these hours.")

results["fusion_on_subset"] = {
    "tabular_this_subset": round(float(tab_here), 4),
    "convlstm_vision": round(float(average_precision_score(test_al["y"], test_al["p_img"])), 4),
    "late_fusion": round(float(average_precision_score(test_al["y"], p_late)), 4),
    "n_rows": int(test_al["n"]), "vision_branch": BEST}

carrying forward: convlstm_time_controlled
aligned val/test rows: 13154/13108

==== Fusion on the n=3-eligible subset ====
branch                              test PR-AUC
Model 1 tabular (this subset)            0.7557
ConvLSTM vision                          0.4268
Late fusion                              0.7303

Model 1 on the full test set is 0.7557; the subset figure above is the
like-for-like comparator for these hours.


The stronger vision branch does not change the outcome: late fusion reaches 0.7303 against 0.7557 for the tabular model on identical hours.

## Section 5 - Save results

In [10]:
import json
from datetime import datetime, timezone

payload = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "ConvLSTM vision branch and diurnal/semidiurnal control",
    "bands": BANDS,
    "n_frames": NF,
    "fairness": "all configs on the n=3-eligible common sample set (same as notebook 07)",
    "reference_channel_stacked": {"n1": 0.4034, "n2": 0.3809, "n3": 0.3638},
    "reference_model1_full_test": 0.7557,
    "notes": ("ConvLSTM trained from scratch; ResNet-18 baseline is ImageNet-pretrained, "
              "so the architectures are not compared on equal footing. Frame spacing "
              "remains hourly."),
    "results": results,
}
out = f"{DRIVE}/models/convlstm_results.json"
json.dump(payload, open(out, "w"), indent=2, default=str)
print("saved:", out)
print(json.dumps({k: v for k, v in results.items()
                  if k != "stratified_by_time_of_day"}, indent=2, default=str))

saved: /content/drive/MyDrive/smart-patio-shield/models/convlstm_results.json
{
  "convlstm_plain": {
    "val_pr_auc": 0.3694,
    "test_pr_auc": 0.4013,
    "channels_per_frame": 2,
    "n_frames": 3
  },
  "convlstm_time_controlled": {
    "val_pr_auc": 0.3744,
    "test_pr_auc": 0.4268,
    "channels_per_frame": 6,
    "n_frames": 3
  },
  "fusion_on_subset": {
    "tabular_this_subset": 0.7557,
    "convlstm_vision": 0.4268,
    "late_fusion": 0.7303,
    "n_rows": 13108,
    "vision_branch": "convlstm_time_controlled"
  }
}
